In [1]:
import os
# os.environ["HTTP_PROXY"] = "http://10.0.0.204:1090"
# os.environ["HTTPS_PROXY"] = "http://10.0.0.204:1090"
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

In [2]:
from PIL import Image
from transformers import AutoTokenizer, AutoModel, AutoImageProcessor, AutoModelForCausalLM
from transformers.generation.configuration_utils import GenerationConfig
from transformers.generation import LogitsProcessorList, PrefixConstrainedLogitsProcessor, UnbatchedClassifierFreeGuidanceLogitsProcessor
import torch

from emu3.mllm.processing_emu3 import Emu3Processor


# model path
EMU_HUB = "/hdd/wangty/diffuser_workdir/emu3/tra_c3-4/Emu3-Stage1-C34-Axial-256_2e-5_12ep"
#"/ssd/wangty/model/Emu3-Gen" # "/hdd/wangty/model/Emu3-Stage1"
VQ_HUB = "/hdd/wangty/model/Emu3-VisionTokenizer"

# prepare model and processor
model = AutoModelForCausalLM.from_pretrained(
    EMU_HUB,
    device_map="cuda:0",
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(EMU_HUB, trust_remote_code=True, padding_side="left")
image_processor = AutoImageProcessor.from_pretrained(VQ_HUB, trust_remote_code=True)
image_tokenizer = AutoModel.from_pretrained(VQ_HUB, device_map="cuda:0", trust_remote_code=True).eval()
processor = Emu3Processor(image_processor, image_tokenizer, tokenizer)



/mnt/nvme_share/wangty/python/anaconda3/envs/emu3/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
from emu3.tokenizer import Emu3VisionVQImageProcessor
import inspect

print(inspect.getfile(Emu3VisionVQImageProcessor))

/home/wangty/github/emu3/Emu3/emu3/tokenizer/image_processing_emu3visionvq.py


In [1]:
from emu3.tokenizer import Emu3VisionVQModel, Emu3VisionVQImageProcessor
image_processor = Emu3VisionVQImageProcessor.from_pretrained("/hdd/wangty/model/Emu3-VisionTokenizer")
print("min_pixels:", image_processor.min_pixels)
print("max_pixels:", image_processor.max_pixels)

/mnt/nvme_share/wangty/python/anaconda3/envs/emu3/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


min_pixels: 1024
max_pixels: 1048576


In [10]:
import json
with open("wty/data/c3_4_test.json","r") as f:
    data=json.load(f)

In [12]:
data[0]

{'name': '02240515201024',
 'text': 'An axial cervical spine MRI image at the C3-4 level. Spinal canal stenosis: stenosis. OPLL: present. Disc protrusion: present. Left foraminal stenosis: none. Right foraminal stenosis: none.',
 'image': '/mnt/nvme_share/wangty/img/tra_256/02240515201024/3-4.png'}

In [15]:
batch_size = 20

batches = []
name_batches = []

for i in range(0, len(data), batch_size):
    batch = data[i:i + batch_size]
    
    # 当前 batch 的 name 列表
    names = [item["name"] for item in batch]
    prompts= [item['text'] for item in batch]
    
    batches.append(batch)
    name_batches.append(names)
print(prompts[0])

An axial cervical spine MRI image at the C3-4 level. Spinal canal stenosis: mild stenosis. OPLL: absent. Disc protrusion: present. Left foraminal stenosis: none. Right foraminal stenosis: none.


In [3]:
# prepare input
POSITIVE_PROMPT = '' #" masterpiece, film grained, best quality."
NEGATIVE_PROMPT = '' #"lowres, bad anatomy, bad hands, text, error, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality, normal quality, jpeg artifacts, signature, watermark, username, blurry."

classifier_free_guidance = 3.0   
# "An axial cervical spine MRI image at the C3-4 level. Spinal canal stenosis: mild stenosis. OPLL: absent. Disc protrusion: present. Left foraminal stenosis: none. Right foraminal stenosis: none."
# "An axial cervical spine MRI image at the C3-4 level. Spinal canal stenosis: none. OPLL: absent. Disc protrusion: none. Left foraminal stenosis: none. Right foraminal stenosis: none."
prompt = "An axial cervical spine MRI image at the C3-4 level. Spinal canal stenosis: mild stenosis. OPLL: absent. Disc protrusion: present. Left foraminal stenosis: none. Right foraminal stenosis: none."
girl="a portrait of young girl."     "a chest X-ray image."
# prompt += POSITIVE_PROMPT
prompt1="An axial cervical spine MRI image at the C3-4 level. Spinal canal stenosis: none. OPLL: absent. Disc protrusion: none. Left foraminal stenosis: none. Right foraminal stenosis: none."
prompts=girl
kwargs = dict(
    mode='G',
    ratio="1:1",
    image_area=65536,    #262144  65536  518400
    return_tensors="pt",
    padding="longest",
)
pos_inputs = processor(text=prompts, **kwargs)
# neg_inputs = processor(text=NEGATIVE_PROMPT, **kwargs)

# prepare hyper parameters
top_k=32
GENERATION_CONFIG = GenerationConfig(
    use_cache=True,
    eos_token_id=model.config.eos_token_id,
    pad_token_id=model.config.pad_token_id,
    max_new_tokens=40960,
    do_sample=True,
    top_k=top_k,
)

h = pos_inputs.image_size[:, 0]
w = pos_inputs.image_size[:, 1]
constrained_fn = processor.build_prefix_constrained_fn(h, w)
logits_processor = LogitsProcessorList([
    # UnbatchedClassifierFreeGuidanceLogitsProcessor(
    #     classifier_free_guidance,
    #     model,
    #     unconditional_ids=neg_inputs.input_ids.to("cuda:0"),
    # ),
    PrefixConstrainedLogitsProcessor(
        constrained_fn ,
        num_beams=1,
    ),
])

# generate
outputs = model.generate(
    pos_inputs.input_ids.to("cuda:0"),
    GENERATION_CONFIG,
    logits_processor=logits_processor,
    attention_mask=pos_inputs.attention_mask.to("cuda:0"),
)

The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.


In [6]:
path=f"test_img/topk_test/{top_k}"
os.makedirs(path, exist_ok=True)
mm_list = [processor.decode(i) for i in outputs]
for idx, item in enumerate(mm_list):
    for i, im in enumerate(item):
        if not isinstance(im, Image.Image):
            continue
        im.save(f"{path}/{idx}.png")

In [5]:
mm_list = processor.decode(outputs[0])
for idx, im in enumerate(mm_list):
        if not isinstance(im, Image.Image):
            continue
        im.save(f"girl.png")

In [4]:
print(pos_inputs)

{'input_ids': tensor([[151849,   2082,  97180,  66727,  34676,  51360,   2168,    518,    279,
            356,     18,     12,     19,   2188,     13,   3089,    977,  38921,
            357,    268,  10704,     25,  23034,    357,    268,  10704,     13,
            506,  62155,     25,  27211,     13,  11735,  80358,   7560,     25,
           3042,     13,  13727,  54534,    977,    357,    268,  10704,     25,
           6857,     13,  10083,  54534,    977,    357,    268,  10704,     25,
           6857,     13, 151852,     18,     17,      9,     18,     17, 151851]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [26]:
processor.decode(torch.tensor([18, 17, 9, 18, 17]))

'32*32'

In [33]:
tokenizer.decode(pos_inputs['input_ids'][0])

'<|extra_203|>An axial cervical spine MRI image at the C3-4 level. Spinal canal stenosis: mild stenosis. OPLL: absent. Disc protrusion: present. Left foraminal stenosis: none. Right foraminal stenosis: none.<|image start|>32*32<|image token|>'

In [5]:
print(model.config.image_area)

262144


In [4]:
outputs.size()

torch.Size([1, 1073])

In [7]:
path = "/mnt/nvme_share/wangty/emu3_data/c3_4_tokenized/feature/01240308210030.pth"
data = torch.load(path, map_location="cpu")

print(type(data))
print(data.keys())

print("name:", data["name"])
print("text:", data["texts"])
print("images type:", type(data["images"]))
print("images shape:", getattr(data["images"], "shape", None))
print("images len:", len(data["images"]))
print("first 20 image tokens:", data["images"][:20])

<class 'dict'>
dict_keys(['name', 'images', 'texts'])
name: 01240308210030
text: An axial cervical spine MRI image at the C3-4 level. Spinal canal stenosis: mild stenosis. OPLL: absent. Disc protrusion: present. Left foraminal stenosis: none. Right foraminal stenosis: none.
images type: <class 'numpy.ndarray'>
images shape: (64, 64)
images len: 64
first 20 image tokens: [[ 8010 12076  4455 ...  4112  8099  5732]
 [ 3089  3749 11358 ...  9684  1596 14245]
 [ 3089  7868  2860 ...  8099  3838 13199]
 ...
 [ 7813  7060  9441 ...  6817  1395  3979]
 [ 6797  2127 10373 ...  7893  3864  8365]
 [12356  2282  7242 ...  6044 14590 13199]]


In [1]:
import torch
path = "/mnt/nvme_share/wangty/emu3_data/c3_4_interleave_tokenized/feature/01240410208103.pth"
data = torch.load(path, map_location="cpu")

print(type(data))
print(data.keys())

print("name:", data["name"])
print("text:", data["texts"])
print("images type:", type(data["image_in"]))
print("images shape:", getattr(data["image_in"], "shape", None))
print("images len:", len(data["image_in"]))
print("images type:", type(data["image_out"]))
print("images shape:", getattr(data["image_out"], "shape", None))
print("images len:", len(data["image_out"]))

<class 'dict'>
dict_keys(['name', 'image_in', 'image_out', 'texts'])
name: 01240410208103
text: <image_in> Given an axial cervical spine MRI image at the C3-4 level, generate a cropped image of the spinal canal region. <image_out>
images type: <class 'numpy.ndarray'>
images shape: (32, 32)
images len: 32
images type: <class 'numpy.ndarray'>
images shape: (8, 8)
images len: 8


/home/wangty/tmp/ipykernel_3940548/707967494.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path, map_location="cpu")
